In [ ]:
import os
import re
import json
import random
import time
import sys
from collections import defaultdict
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

# ===================================================================
# 定价者：顶点练习（第 6 周）
# THE PRICER: CAPSTONE EXERCISE (WEEK 6)
# ===================================================================
# 该脚本通过以下方式改进了讲师的第 5 天微调模型：
# This script improves upon the instructor's Day 5 fine-tuned model by:
# 1. 平衡抽样：确保法学硕士了解 500 美元的电视，而不仅仅是 5 美元的电缆。
# 1. Balanced Sampling: Ensuring the LLM learns about $500 TVs, not just $5 cables.
# 2. 专家角色：为人工智能设计特定领域的提示。
# 2. Expert Persona: Engineering a domain-specific prompt for the AI.


## # 第 0 步：设置和导入

## # 从根目录加载 .env 文件

In [ ]:
load_dotenv(override=True)

# 将 week6 文件夹添加到 Python 路径中，以便我们可以导入讲师的自定义课程
# Add the week6 folder to the Python path so we can import the instructor's custom classes
sys.path.append(os.path.abspath("../../../week6"))
try:
    from pricer.items import Item
    from pricer.evaluator import evaluate, Tester
except ImportError:
    print("Warning: Could not import pricer modules. Make sure you are running this from the community-contributions folder.")

# 初始化客户端
# Initialize clients
openai = OpenAI()
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(hf_token, add_to_git_credential=True)
else:
    print("WARNING: HF_TOKEN not found in .env")


## # 第 1 步：加载精简数据集

## print("\n--- 第 1 步：加载数据集 ---")

In [ ]:
# 我们明确地从 HuggingFace 中“借用”了完美清理的第一天数据
# We explicitly "borrow" the perfectly cleaned Day 1 data from HuggingFace
username = "ed-donner"
dataset_name = f"{username}/items_lite"

# 测试=~1000 个项目，val=~1000 个项目，训练=~20000 个项目
# test=~1000 items, val=~1000 items, train=~20000 items
train, val, test = Item.from_hub(dataset_name)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


## # 第 2 步：第一个改进 - 平衡采样

In [ ]:
# 缺陷：如果前 100 件商品都是廉价手机壳，则型号
# The Flaw: If the first 100 items are all cheap phone cases, the model
# 永远不知道昂贵的物品是什么样子。
# never learns what an expensive item looks like.
# 解决办法：我们将价格分为不同的桶，并从每个桶中平均提取。
# The Fix: We categorize prices into buckets, and pull equally from each bucket.

def categorize_price(price):
    if price < 50:
        return '$0-50'
    elif price < 150:
        return '$50-150'
    elif price < 300:
        return '$150-300'
    else:
        return '$300+'

print("\n--- STEP 2: Creating Balanced Training Data ---")
# 将项目分组到桶中
# Group items into buckets
price_buckets = defaultdict(list)
for item in train:
    bucket = categorize_price(item.price)
    price_buckets[bucket].append(item)

# 我们需要 100 个项目进行微调。由于我们有 4 个桶，因此我们从每个桶中取出 25 个。
# We want 100 items for fine-tuning. Since we have 4 buckets, we take 25 from each.
ITEMS_PER_BUCKET = 25 
fine_tune_train = []
random.seed(42)

for bucket, items_in_bucket in price_buckets.items():
    # 从此存储桶中随机选择项目（或尽可能多的项目）
    # Randomly select items from this bucket (or as many as there are)
    sample_size = min(ITEMS_PER_BUCKET, len(items_in_bucket))
    sample = random.sample(items_in_bucket, sample_size)
    fine_tune_train.extend(sample)

# 对它们进行洗牌，以便它们在训练期间不会按价格完美分组
# Shuffle them so they aren't grouped perfectly by price during training
random.shuffle(fine_tune_train)

# 我们将随机抽取 50 个项目作为验证集
# We will take 50 random items for the validation set
# 我们还需要一个平衡的 50 项验证集（每个桶中大约 12 或 13 项）
# We also want a balanced 50-item validation set (approx 12 or 13 from each bucket)
val_buckets = defaultdict(list)
for item in val:
    bucket = categorize_price(item.price)
    val_buckets[bucket].append(item)

fine_tune_validation = []
for bucket, items_in_bucket in val_buckets.items():
    sample = random.sample(items_in_bucket, min(13, len(items_in_bucket)))
    fine_tune_validation.extend(sample)
fine_tune_validation = fine_tune_validation[:50] # Trim exactly to 50

print(f"Created a balanced training set of {len(fine_tune_train)} items.")
for bucket in sorted(price_buckets.keys()):
    count = sum(1 for x in fine_tune_train if categorize_price(x.price) == bucket)
    print(f" - {bucket} bucket: {count} items")


## # 第 3 步：第二次改进 - 专家角色

In [ ]:
# 修复：我们将系统提示升级为具有域上下文的“专家角色”。
# The Fix: We upgrade the System Prompt to an "Expert Persona" with domain context.

EXPERT_SYSTEM_PROMPT = """You are a senior pricing analyst with deep expertise in consumer electronics, appliances, and retail goods.
Analyze the brand, product specifications, and features to estimate the most likely retail market price in USD.
Respond ONLY with the format: Price is $XX.XX"""

def messages_for(item):
    """Creates the 'Flashcard' for the LLM. Front = Description, Back = Price"""
    # 清理导师乱七八糟的test_prompt
    # Clean up the instructor's messy test_prompt
    user_prompt = item.test_prompt().replace(" to the nearest dollar", "").replace("\n\nPrice is $", "")
    return [
        {"role": "system", "content": EXPERT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": f"Price is ${item.price:.2f}"} # The answer on the back
    ]


## # 第 4 步：格式化并保存 JSONL 文件

In [ ]:
def make_jsonl(items):
    result = ""
    for item in items:
        # 将字典转换为 JSON 字符串
        # Convert the dictionary into a JSON string
        result += '{"messages": ' + json.dumps(messages_for(item)) + '}\n'
    return result.strip()

def write_local(items, filename):
    # 将文件写入本地
    # Write the file locally
    with open(filename, "w") as f:
        f.write(make_jsonl(items))
    return os.path.abspath(filename)

train_path = write_local(fine_tune_train, "fine_tune_train.jsonl")
val_path = write_local(fine_tune_validation, "fine_tune_validation.jsonl")

print(f"✅ Successfully wrote structured training data to:\n -> {train_path}")
print(f"✅ Successfully wrote structured validation data to:\n -> {val_path}")
